<a href="https://colab.research.google.com/github/nayem9b/3D-Portfolio/blob/main/Lets_Build_From_Ashes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import random
import cv2
import torchvision.models as models
import pandas as pd
import numpy as np
import matplotlib.gridspec as gridspec
import seaborn as sns
import zlib
import itertools
import sklearn
import itertools
import scipy

import skimage
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import csv
import warnings
import keras
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np

!pip install deap
from deap import base, creator, tools, algorithms
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, hinge_loss
from pathlib import Path
from skimage.transform import resize
from tqdm import tqdm
from sklearn import model_selection
from sklearn.model_selection import train_test_split, KFold, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix, make_scorer, accuracy_score, classification_report
from keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D, Lambda, MaxPool2D, BatchNormalization
# from keras.preprocessing.image import ImageDataGenerator
from keras import models, layers, optimizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.utils import class_weight
from keras.optimizers import SGD, RMSprop, Adam, Adagrad, Adadelta, RMSprop
from keras.models import Sequential, model_from_json
from keras.layers import Activation,Dense, Dropout, Flatten, Conv2D, MaxPool2D
from keras.layers import MaxPooling2D,AveragePooling2D, GlobalAveragePooling2D,BatchNormalization
from keras.utils import array_to_img, img_to_array, load_img
from glob import glob
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from tensorflow.keras import mixed_precision
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from deap import base, creator, tools, algorithms
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, hinge_loss
from pathlib import Path
%matplotlib inline
!pip install torchvision
warnings.filterwarnings("ignore")
mixed_precision.set_global_policy('mixed_float16')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 3.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Have mounted the drive to Colab**

In [3]:
normal_dir = '/content/drive/MyDrive/all_denoised_document/Speckle Noise/Noised/CNV'
dme_dir = '/content/drive/MyDrive/all_denoised_document/Speckle Noise/Noised/DME'
drusen_dir = '/content/drive/MyDrive/all_denoised_document/Speckle Noise/Noised/DRUSEN'
cnv_dir = '/content/drive/MyDrive/all_denoised_document/Speckle Noise/Noised/NORMAL'
image_size = (256, 256) # Image dimensions are set to 256*256
num_classes = 4

**The below function load_and_preprocess_images loads PNG images from a specified directory, resizes them to 256x256 pixels, normalizes the pixel values, and associates each image with a given label.**

In [4]:
def load_and_preprocess_images(image_dir, label):
    images = []
    labels = []
    for filename in os.listdir(image_dir):
        if filename.endswith('.png'):
            img = load_img(os.path.join(image_dir, filename), target_size=image_size)
            img_array = img_to_array(img)
            img_array /= 255.0  # Normalize pixel values into 0 and 1. 8 bit color channel division.
            images.append(img_array)
            labels.append(label)
    return images, labels

**The below code loads and preprocesses images from each directory by calling the previous function, assigning corresponding labels (NORMAL, DME, CNV, DRUSEN).**

In [5]:
normal_images, normal_labels = load_and_preprocess_images(normal_dir, label='NORMAL')
dme_images, dme_labels = load_and_preprocess_images(dme_dir, label='DME')
cnv_images, cnv_labels = load_and_preprocess_images(cnv_dir, label='CNV')
drusen_images, drusen_labels = load_and_preprocess_images(drusen_dir, label='DRUSEN')

**The below code prints the length of the loaded images**

In [6]:
print("Normal_images: ", len(normal_images))
print("DME_images: ", len(dme_images))
print("CNV_images: ", len(cnv_images))
print("Drusen_images: ", len(drusen_images))

Normal_images:  1000
DME_images:  1000
CNV_images:  1000
Drusen_images:  1000


**The code combines the image datasets from all categories into a single array X and creates a corresponding label array y for each image. The labels are repeated according to the number of images in each category.**

In [7]:
# Combine datasets
X = np.array(normal_images + dme_images + cnv_images + drusen_images)
y = np.array(['NORMAL'] * len(normal_images) + ['DME'] * len(dme_images) +
             ['CNV'] * len(cnv_images) + ['DRUSEN'] * len(drusen_images))  #for labeling purpose array

**The below code converts the string labels in y to numeric values by mapping each label to its index in the class_names list, resulting in the y_numeric array.**

In [8]:
# Convert labels to numeric values
class_names = ['NORMAL', 'DME', 'CNV', 'DRUSEN']
y_numeric = np.array([class_names.index(label) for label in y])

In [9]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np


# Load the pretrained EfficientNet model (EfficientNet-B0 variant)
efficientnet = models.efficientnet_b0(pretrained=True)

# Remove the final classification layer to use the model as a feature extractor
# This keeps the convolutional layers and removes the fully connected output layer
efficientnet = nn.Sequential(*list(efficientnet.children())[:-1])

# Set the model to evaluation mode, disabling certain layers like dropout and batch normalization
efficientnet.eval()

# Define a transform to preprocess input images for the EfficientNet model
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize the image to 224x224 pixels
    transforms.ToTensor(),          # Convert image to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize based on ImageNet stats
])


# Function to extract features from an image using the EfficientNet model
def extract_features(image_path, model):
    # Open the image, convert to RGB (in case it's in a different mode)
    image = Image.open(image_path).convert('RGB')

    # Apply the defined preprocessing transforms
    image = transform(image).unsqueeze(0)  # Add a batch dimension

    # Pass the image through the model (without updating weights)
    with torch.no_grad():  # No need to track gradients for feature extraction
        features = model(image)

    # Squeeze the output to remove unnecessary dimensions and return as a NumPy array
    return features.squeeze().numpy()


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 126MB/s] 


**Split the dataset into training (80%) and testing (20%)**

In [10]:
# One-hot encode labels
# Converts the numeric labels (y_numeric) into one-hot encoded vectors,
# where each label is represented as a binary vector of length equal to the number of classes.
y_encoded = to_categorical(y_numeric, num_classes=len(class_names))

# Split the dataset into training (80%) and testing (20%)
# The train_test_split function randomly splits the data, maintaining the same label distribution (stratify) across both sets.
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_numeric)

# Output the number of training and testing samples
# Printing the size of the training and testing sets to verify the split.
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 3200
Testing samples: 800


In [11]:
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling2D, Multiply, Reshape

# Efficient Channel Attention (ECA) block
class ECABlock(tf.keras.layers.Layer):
    def __init__(self, kernel_size=3):
        # Initializes the ECA block with the specified kernel size (default is 3)
        super(ECABlock, self).__init__()

        # Global average pooling to compute a global descriptor of the input feature map
        self.global_avg_pool = GlobalAveragePooling2D()

        # 1D convolution layer with a kernel of size 'kernel_size' (used to capture local dependencies)
        # The convolution layer uses 'sigmoid' activation to generate attention weights
        self.conv1d = Conv1D(1, kernel_size=kernel_size, padding="same", activation="sigmoid")

    def call(self, inputs):
        # Perform global average pooling along the spatial dimensions (height, width)
        avg_pool = self.global_avg_pool(inputs)

        # Expand the dimensions of avg_pool to match the input tensor shape (for further processing)
        avg_pool = tf.expand_dims(avg_pool, axis=-1)

        # Apply 1D convolution to the pooled feature, generating attention weights
        conv1d = self.conv1d(avg_pool)

        # Reshape the output of the convolution to match the input feature map's channel dimension
        conv1d = Reshape((1, 1, inputs.shape[-1]))(conv1d)

        # Multiply the input feature map with the generated attention weights to apply the attention
        return Multiply()([inputs, conv1d])


In [12]:
def extract_features(image_path, model):
    # Open the image from the provided path and convert it to RGB (in case it is in a different color mode)
    image = Image.open(image_path).convert('RGB')

    # Apply the defined transformations (resize, convert to tensor, normalize) and add a batch dimension (unsqueeze)
    image = transform(image).unsqueeze(0)

    # Pass the image through the model to extract features, ensuring no gradient tracking is done (for efficiency)
    with torch.no_grad():
        features = model(image)

    # Remove unnecessary dimensions (squeeze) and return the features as a NumPy array
    return features.squeeze().numpy()


In [13]:
# Not using I Guess

def build_model():
    # Load the EfficientNetB0 model pre-trained on ImageNet, excluding the top classification layer.
    # The input shape is set to (224, 224, 3) for color images (RGB).
    base_model = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224, 224, 3))

    # Take the output of the base model (the feature maps) and pass it through the ECA block
    x = base_model.output
    x = ECABlock()(x)  # Apply Efficient Channel Attention (ECA) block

    # Apply Global Average Pooling to reduce the feature map to a single vector
    x = GlobalAveragePooling2D()(x)

    # Add a fully connected layer with 256 units and ReLU activation for more abstraction
    x = tf.keras.layers.Dense(256, activation="relu")(x)

    # Add a Dropout layer with a rate of 50% to prevent overfitting
    x = tf.keras.layers.Dropout(0.5)(x)

    # Output layer with 4 units (for the 4 classes) and softmax activation to predict class probabilities
    x = tf.keras.layers.Dense(4, activation="softmax")(x)

    # Create the model by specifying the input (base_model.input) and output layers (x)
    model = tf.keras.Model(inputs=base_model.input, outputs=x)

    # Compile the model with Adam optimizer and categorical cross-entropy loss (for multi-class classification)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model



**The below code defines a feature selection problem that maximizes accuracy and minimizes feature count, using pymoo for optimization. The evaluation function simulates accuracy and counts selected features.**

In [14]:
# Install the pymoo library for multi-objective optimization
!pip install pymoo

from pymoo.core.problem import Problem
import numpy as np

# Define the feature selection problem as a subclass of pymoo's Problem
class FeatureSelectionProblem(Problem):
    def __init__(self, X, y):
        # Initialize the problem with the number of features (n_var),
        # the number of objectives (n_obj = 2: accuracy and number of features),
        # and the bounds for feature selection (xl=0, xu=1 indicates binary selection).
        super().__init__(n_var=X.shape[1], n_obj=2, xl=0, xu=1)
        self.X = X  # The feature matrix (input data)
        self.y = y  # The labels (output data)

    def _evaluate(self, x, out, *args, **kwargs):
        # Convert the binary vector x into selected features (1 for selected, 0 for not selected)
        selected_features = np.where(x > 0.5, 1, 0)

        # Reduce the input data to the selected features
        reduced_X = self.X[:, selected_features.sum(axis=0) > 0]

        # Simulate accuracy as a random value between 0.7 and 0.95 (replace with actual model evaluation)
        acc = np.random.uniform(0.7, 0.95)

        # Count the number of selected features
        n_features = np.sum(selected_features)

        # Set the objectives: maximize accuracy (hence the negative sign) and minimize the number of features
        out["F"] = np.column_stack([-acc, n_features])  # Output the objectives



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 7.7 MB/s eta 0:00:00
  Created wheel for grapheme: filename=grapheme-0.6.0-py3-none-any.whl size=210082 sha256=051262adba5c60376ee45abf2ab2308d0865e9b750793d311d967352d7e65026
  Stored in directory: /root/.cache/pip/wheels/ee/3b/0b/1b865800e916d671a24028d884698674138632a83fdfad4926
Successfully built grapheme


**The below code generates training and validation data from a directory, extracts features from images using EfficientNet, and stores them in arrays. It then prints the shape of the feature matrix and the number of extracted features per image.**

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataset_dir = "/content/drive/MyDrive/all_denoised_document/Speckle Noise/Noised"

# Initialize the ImageDataGenerator with rescaling and validation split
train_datagen = ImageDataGenerator(rescale=1.0/255.0, validation_split=0.2)

# Set up the training data generator, loading images and performing rescaling
train_generator = train_datagen.flow_from_directory(
    dataset_dir, target_size=(224, 224), batch_size=32, class_mode='categorical', subset='training'
)

# Set up the validation data generator, loading images and performing rescaling
val_generator = train_datagen.flow_from_directory(
    dataset_dir, target_size=(224, 224), batch_size=32, class_mode='categorical', subset='validation'
)

# List all the class directories (labels) in the dataset directory
classes = os.listdir(dataset_dir)

# Initialize lists to store extracted features and their corresponding labels
features_list = []
labels_list = []

# Loop through each class (label) and its corresponding images
for label, class_dir in enumerate(classes):
    class_path = os.path.join(dataset_dir, class_dir)

    # Loop through each image in the class directory
    for image_name in os.listdir(class_path):
        image_path = os.path.join(class_path, image_name)

        # Extract features from each image using the EfficientNet model
        features = extract_features(image_path, efficientnet)

        # Append the extracted features and their corresponding label to the lists
        features_list.append(features)
        labels_list.append(label)

# Stack the list of feature arrays into a single 2D array
features = np.vstack(features_list)

# Stack the list of label arrays into a single 1D array
labels = np.hstack(labels_list)

# Print the shape of the feature array (number of samples and feature dimensions)
print(f'Features shape: {features.shape}')

# Print the number of features extracted per image (the second dimension of the feature array)
print(f'Number of features extracted per image: {features.shape[1]}')


Found 3200 images belonging to 4 classes.
Found 800 images belonging to 4 classes.


In [ ]:
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.problems import get_problem
from pymoo.operators.crossover.pntx import TwoPointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.optimize import minimize
from pymoo.visualization.scatter import Scatter
import joblib
import multiprocessing

In [ ]:
# Initialize lists to store accuracy and loss
train_accuracy_svm_list = []
validation_accuracy_svm_list = []
train_loss_svm_list = []
validation_loss_svm_list = []
selected_feature_counts = []  # List to store feature counts after selection

class FeatureSelectionProblem(Problem):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
        super().__init__(n_var=features.shape[1], n_obj=1, xl=0, xu=1, type_var=np.bool_)

    def _evaluate(self, x, out, *args, **kwargs):
        res = []
        for individual in x:
            selected_features = np.where(individual)[0]
             # Count and store the number of selected features for each individual
            selected_feature_counts.append(len(selected_features))

            if len(selected_features) == 0:
                res.append([0])  # Invalid solution
                continue
            X_train, X_test, y_train, y_test = train_test_split(self.features[:, selected_features], self.labels, test_size=0.3, random_state=42)

            # SVM
            svm = SVC(kernel='poly')
            svm.fit(X_train, y_train)

            # Calculate training accuracy
            y_train_pred = svm.predict(X_train)
            train_accuracy_svm = accuracy_score(y_train, y_train_pred)
            train_accuracy_svm_list.append(train_accuracy_svm * 100)
            #print(f"Train Accuracy SVM: {train_accuracy_svm * 100:.2f}%")

            # Calculate validation accuracy
            y_test_pred = svm.predict(X_test)
            validation_accuracy_svm = accuracy_score(y_test, y_test_pred)
            validation_accuracy_svm_list.append(validation_accuracy_svm * 100)
            #print(f"Validation Accuracy SVM: {validation_accuracy_svm * 100:.2f}%")

            # Calculate training and validation loss
            train_loss_svm = hinge_loss(y_train, svm.decision_function(X_train))
            validation_loss_svm = hinge_loss(y_test, svm.decision_function(X_test))
            train_loss_svm_list.append(train_loss_svm)
            validation_loss_svm_list.append(validation_loss_svm)
            #print(f"Train Loss SVM: {train_loss_svm * 100:.2f}")
            #print(f"Validation Loss SVM: {validation_loss_svm * 100:.2f}")

            res.append([1 - validation_accuracy_svm])

        out['F'] = np.array(res)



In [ ]:
problem = FeatureSelectionProblem(features, labels)

In [ ]:
!pip install deap
from deap import base, creator, tools, algorithms

# Parallel processing setup
pool = multiprocessing.Pool()
toolbox = base.Toolbox()
toolbox.register("map", pool.map)

In [ ]:
# Define the NSGA-II algorithm with specified settings
algorithm = NSGA2(
    pop_size=25,  # Set the population size to 25 individuals in each generation
    sampling=BinaryRandomSampling(),  # Use binary random sampling to create the initial population
    crossover=TwoPointCrossover(),  # Apply two-point crossover during reproduction (combining parents' solutions)
    mutation=BitflipMutation(),  # Use bit-flip mutation to introduce diversity by randomly flipping bits
    eliminate_duplicates=True  # Remove duplicate solutions to ensure unique candidates in the population
)

In [ ]:
# Minimize the feature selection problem using the specified algorithm
res = minimize(problem,  # The problem to solve (e.g., feature selection)
               algorithm,  # The optimization algorithm to use (e.g., NSGA-II)
               ('n_gen', 25),  # Set the number of generations (iterations) to 25
               seed=1,  # Set the random seed for reproducibility
               save_history=True,  # Save the history of the optimization process
               verbose=True,  # Print detailed output of the optimization process
               eliminate_duplicates=True)  # Remove duplicate solutions during optimization

In [ ]:
# After optimization, print the feature counts
print("Selected feature counts after NSGA-II for each solution:")
for i, count in enumerate(selected_feature_counts, start=1):
    print(f"Solution {i}: {count} features selected")

In [ ]:
# Print selected feature counts per image
print("Number of selected features for each image after NSGA-II:")
for i, count in enumerate(selected_feature_counts, start=1):
    print(f"Image {i}: {count} features selected")

In [ ]:
!pip install tabulate

# Print the results as a table
from tabulate import tabulate
table = [
    [i+1, train_accuracy, val_accuracy, train_loss, val_loss]
    for i, (train_accuracy, val_accuracy, train_loss, val_loss)
    in enumerate(zip(train_accuracy_svm_list, validation_accuracy_svm_list, train_loss_svm_list, validation_loss_svm_list))
]
headers = ["Iteration", "Training Accuracy (%)", "Validation Accuracy (%)", "Training Loss", "Validation Loss"]
print(tabulate(table, headers=headers, tablefmt="grid"))

In [ ]:
import matplotlib.pyplot as plt

# Set font size and weight for all text in the plot
plt.rcParams.update({
    'font.size': 12,      # Font size
    'font.weight': 'bold' # Font weight
})

# Plot training and validation accuracy
plt.figure(figsize=(12, 4))

# Plot Training and Validation Accuracy
plt.subplot(1, 2, 1)
plt.plot(train_accuracy_svm_list, label='Training Accuracy', color='red')
plt.plot(validation_accuracy_svm_list, label='Validation Accuracy', color='blue')
plt.ylim(80, 104)  # Set y-axis range from 80 to 100
plt.yticks(np.arange(84, 102, 2))  # Step size of 2 for y-axis ticks
plt.xlabel('Iteration', fontweight='bold')
plt.ylabel('Accuracy (%)', fontweight='bold')
plt.title('Training and Validation Accuracy', fontweight='bold')
plt.legend()
#plt.yticks(range(70, len(train_accuracy_svm_list) + 1, 20))  # Set x-axis ticks with step size of 20

# Plot Training and Validation Loss
plt.subplot(1, 2, 2)
#plt.figure(figsize=(12, 5))
plt.plot(train_loss_svm_list, label='Training Loss', color='red')
plt.plot(validation_loss_svm_list, label='Validation Loss', color='blue')
plt.xlabel('Iteration', fontweight='bold')
plt.ylabel('Loss', fontweight='bold')
plt.title('Training and Validation Loss', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Import SVM and metrics from scikit-learn
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

# For plotting
import matplotlib.pyplot as plt


# Assuming X_train, X_test, y_train, y_test are already defined

# Function to plot ROC curve
def plot_roc_curve(fpr, tpr, roc_auc, title):
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.show()

# Train and evaluate SVM
svm = SVC(kernel='poly', probability=True)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
y_score_svm = svm.decision_function(X_test)

# Compute metrics for SVM
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_score_svm)
roc_auc_svm = auc(fpr_svm, tpr_svm)

print(f'SVM Accuracy: {accuracy_svm:.4f}')
print(f'SVM Precision: {precision_svm:.4f}')
print(f'SVM Recall: {recall_svm:.4f}')
print(f'SVM F1 Score: {f1_svm:.4f}')
print(f'SVM AUC: {roc_auc_svm:.4f}')

# Plot ROC curve for SVM
plot_roc_curve(fpr_svm, tpr_svm, roc_auc_svm, 'SVM ROC Curve')

# Confusion matrix for SVM
cm_svm = confusion_matrix(y_test, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm)
disp_svm.plot()
plt.title('SVM Confusion Matrix')
plt.show()